In [ ]:
import pandas as pd
from utils import json_to_df, read_jsons

In [2]:
BASE_PATH = "/home/rafael.jarczewski/selection-agent/results/sbrc/mnist/exp1-literatura/{selection_algorithm}/{sample_size}"

In [3]:
SELECTION_ALGORTIHM = ['poc', 'oort', 'random', 'rrobin']
SAMPLE_SIZES = [15, 10, 5, 'rnd']

In [4]:
dfs = []
for sa in SELECTION_ALGORTIHM:
    for ssize in SAMPLE_SIZES:
        js_list = read_jsons(BASE_PATH.format(selection_algorithm=sa, sample_size=ssize))
        if len(js_list) > 0:
            df = pd.concat([json_to_df(js) for js in js_list])
            df['sample_size'] = str(ssize)
            dfs.append(df)

In [5]:
data = pd.concat(dfs)

In [6]:
data['hue_column'] = data['selection_algorithm'] +'-'+data['sample_size']
data = data.loc[data['round'] > 1].sort_values('hue_column')

In [8]:
data = data.loc[data['round'] > 1].sort_values('hue_column')
avg_accuracy = data.groupby(by=['hue_column', 'round'])[['eval_accuracy', 'sample_time']].mean().reset_index()
std_accuracy = data.groupby(by=['hue_column', 'round'])[['eval_accuracy', 'sample_time']].std().reset_index()
idxmax = avg_accuracy.groupby(['hue_column'])['eval_accuracy'].idxmax()

In [9]:
avg_sample_time = data.groupby(by=['hue_column', 'round'])['sample_time'].mean().reset_index()

In [10]:
selected = data.loc[data['selected']]
amount_selected = (selected.groupby(['round', 'hue_column']).size()/3).reset_index()
amount_selected = amount_selected.loc[amount_selected['round'] > 1]

In [11]:
total_params = 62006
size_mb = 0.24

In [12]:
selected = data.loc[data['selected']]
amount_selected = (selected.groupby(['round', 'hue_column']).size()/3).reset_index()
amount_selected = amount_selected.loc[amount_selected['round'] > 1]
amount_selected['download_mb'] = amount_selected[0]*size_mb

In [13]:
d = pd.DataFrame({
    "sel":amount_selected.groupby(['hue_column']).sum().reset_index()['hue_column'],
    "accuracy": avg_accuracy.groupby(['hue_column'])['eval_accuracy'].max().reset_index()['eval_accuracy'],
    "std_accuracy": std_accuracy.iloc[idxmax].reset_index()['eval_accuracy'],
    "k_medio": amount_selected.groupby(['hue_column']).mean().reset_index()[0],
    "k_std": amount_selected.groupby(['hue_column']).std().reset_index()[0],
    "download_mb": amount_selected.groupby(['hue_column']).sum().reset_index()['download_mb'],
    "sample_time": avg_sample_time.groupby("hue_column").mean().reset_index()['sample_time']
})

In [ ]:
mask = d['sel'].apply(lambda x: 'rrobin' in x)
# d.loc[mask]

,sel,accuracy,std_accuracy,k_medio,k_std,download_mb,sample_time
12,rrobin-10,0.919467,0.103269,10.000000,0.000000,117.60,0.000215
13,rrobin-15,0.947467,0.059776,15.000000,0.000000,176.40,0.000195
14,rrobin-5,0.922133,0.098726,5.000000,0.000000,58.80,0.000217
15,rrobin-rnd,0.961867,0.024807,13.802721,3.549203,162.32,0.000229


In [15]:
d.to_csv('./data/normal_sel.csv', index=False)